# Interfacial Energy

Calculate the interfacial energy γ (eV/Å²) of a heterostructure using a DFT workflow on the Mat3ra platform.

This notebook loads an interface from `../uploads`, strips atom labels (required for Quantum ESPRESSO), resolves substrate and film bulk materials from `metadata.build` crystal hashes, and checks that both bulk materials and their `total_energy` properties already exist on the platform.

If substrate bulk, film bulk, or either bulk total energy is missing, the notebook stops and points to the [Total Energy](total_energy.ipynb) notebook. It does not create or run bulk pre-calculations automatically.

The job uses a single interface material. The workflow resolves bulks and fetches their precomputed total energies automatically.

<h2 style="color:green">Usage</h2>

1. Build an interface (for example via `create_interface_with_min_strain_zsl.ipynb`) and export it to `../uploads`, or set `INTERFACE_NAME` to match an existing interface on the platform.
2. Run [Total Energy](total_energy.ipynb) for each resolved substrate and film bulk material.
3. Set parameters in cells 1.2 and 1.3 below.
4. Click "Run" > "Run All".
5. Inspect resolved bulks, bulk total energies, and the final interfacial energy.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load interface, strip labels, resolve substrate/film bulks, and verify they exist.
4. Verify bulk total energies exist.
5. Configure and save the Interfacial Energy workflow.
6. Configure compute.
7. Create, submit, and monitor the job.
8. Retrieve results.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"  # Name to search in uploads or on the platform

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "interfacial_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Interfacial Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


### 1.3. Set specific interfacial energy parameters

In [ ]:
# K-grid for interface SCF (if not set, KPPRA is used by default)
SCF_KGRID = None  # e.g., [4, 4, 1]

## 2. Authenticate and initialize API client
### 2.1. Authenticate


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account to work under


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load interface and resolve substrate/film bulks
### 3.1. Load interface from local file or platform


In [ ]:
import re
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

interface = load_material_from_folder(FOLDER, INTERFACE_NAME)

if interface is None:
    interface_matches = client.materials.list({
        "name": {"$regex": re.escape(INTERFACE_NAME), "$options": "i"},
    })
    if not interface_matches:
        raise ValueError(
            f"No interface containing '{INTERFACE_NAME}' was found in '{FOLDER}' or on the platform."
        )
    interface = Material.create(interface_matches[0])
    print(f"♻️  Loaded interface from platform: {interface_matches[0]['_id']}")
else:
    print(f"✅ Loaded interface from folder: {interface.name}")

visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")


### 3.2. Prepare interface for Quantum ESPRESSO
Strip atom labels. Labels are useful for analysis notebooks but are not compatible with QE input generation.


In [ ]:
interface_material = interface.clone()
interface_material.basis.set_labels_from_list([])
print(f"Prepared interface material: {interface_material.name}")
visualize(interface_material, repetitions=[1, 1, 1], rotation="-90x")


### 3.3. Load workflow from Standata


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

interfacial_workflow_config = WorkflowStandata.filter_by_application(APPLICATION_NAME).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)
interfacial_workflow = Workflow.create(interfacial_workflow_config)
resolve_refs_subworkflow = interfacial_workflow.subworkflows[0]
fetch_substrate_te_subworkflow = interfacial_workflow.subworkflows[1]
fetch_film_te_subworkflow = interfacial_workflow.subworkflows[2]
print(f"Loaded workflow: {interfacial_workflow.name}")

### 3.4. Resolve substrate bulk from interface build metadata

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.analysis import get_interface_bulk_crystal
from mat3ra.notebooks_utils.ui import display_JSON

CRYSTAL_SUBSTRATE = get_interface_bulk_crystal(interface_material, part="substrate")
io_unit = resolve_refs_subworkflow.get_unit_by_name(name="io-bulk-substrate")
query_template = io_unit.input[0]["endpoint_options"]["params"]["query"]
projection_template = io_unit.input[0]["endpoint_options"]["params"]["projection"]

substrate_query = eval(query_template, {"__builtins__": {}}, {"CRYSTAL_SUBSTRATE": CRYSTAL_SUBSTRATE})
projection = eval(projection_template, {"__builtins__": {}})
substrate_matches = client.materials.list(query=substrate_query, projection=projection)
if not substrate_matches:
    raise ValueError(
        "The substrate bulk material resolved from interface metadata is not on the platform. "
        "Run total_energy.ipynb for that bulk material first, then rerun this notebook."
    )

substrate_bulk_response = substrate_matches[0]
substrate_bulk = Material.create(substrate_bulk_response)
BULK_SUBSTRATE_MATERIAL_IDS = [material["_id"] for material in substrate_matches]
print(f"Found substrate bulk material: {substrate_bulk_response['_id']}")
print(f"Resolved substrate bulk query: {substrate_query}")
display_JSON({"substrate_query": substrate_query}, level=2)
visualize(substrate_bulk)


### 3.5. Resolve film bulk from interface build metadata


In [ ]:
CRYSTAL_FILM = get_interface_bulk_crystal(interface_material, part="film")
io_unit = resolve_refs_subworkflow.get_unit_by_name(name="io-bulk-film")
query_template = io_unit.input[0]["endpoint_options"]["params"]["query"]
projection_template = io_unit.input[0]["endpoint_options"]["params"]["projection"]

film_query = eval(query_template, {"__builtins__": {}}, {"CRYSTAL_FILM": CRYSTAL_FILM})
projection = eval(projection_template, {"__builtins__": {}})
film_matches = client.materials.list(query=film_query, projection=projection)
if not film_matches:
    raise ValueError(
        "The film bulk material resolved from interface metadata is not on the platform. "
        "Run total_energy.ipynb for that bulk material first, then rerun this notebook."
    )

film_bulk_response = film_matches[0]
film_bulk = Material.create(film_bulk_response)
BULK_FILM_MATERIAL_IDS = [material["_id"] for material in film_matches]
print(f"Found film bulk material: {film_bulk_response['_id']}")
print(f"Resolved film bulk query: {film_query}")
display_JSON({"film_query": film_query}, level=2)
visualize(film_bulk)


### 3.6. Save the interface material for the workflow


In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_interface_response = get_or_create_material(client, interface_material, ACCOUNT_ID)
saved_interface = Material.create(saved_interface_response)
print(f"✅ Saved interface material: {saved_interface_response['_id']}")


## 4. Check bulk total energies
### 4.1. Check substrate bulk total energy


In [ ]:
# Extract the substrate bulk TE queries and projections from the workflow definition
# so they stay in sync with what the workflow actually uses at runtime.
io_job_unit = fetch_substrate_te_subworkflow.get_unit_by_name(name="io-bulk-te-job-substrate")
job_query_template = io_job_unit.input[0]["endpoint_options"]["params"]["query"]
job_projection_template = io_job_unit.input[0]["endpoint_options"]["params"]["projection"]

BULK_MATERIAL_IDS = BULK_SUBSTRATE_MATERIAL_IDS
job_query = eval(job_query_template, {"__builtins__": {}}, {"BULK_MATERIAL_IDS": BULK_MATERIAL_IDS})
job_projection = eval(job_projection_template, {"__builtins__": {}})
jobs = client.jobs.list(query=job_query, projection=job_projection)
if not jobs:
    raise RuntimeError(
        "Substrate bulk total energy not found. Run total_energy.ipynb for the substrate bulk first."
    )

BULK_TE_JOB_ID = jobs[0]["_id"]
io_te_unit = fetch_substrate_te_subworkflow.get_unit_by_name(name="io-te-bulk-substrate")
te_query_template = io_te_unit.input[0]["endpoint_options"]["params"]["query"]
te_projection_template = io_te_unit.input[0]["endpoint_options"]["params"]["projection"]

te_query = eval(te_query_template, {"__builtins__": {}}, {"BULK_TE_JOB_ID": BULK_TE_JOB_ID})
te_projection = eval(te_projection_template, {"__builtins__": {}})
props = client.properties.list(query=te_query, projection=te_projection)
substrate_te_property = props[0] if props else None
if substrate_te_property is None:
    raise RuntimeError(
        "Substrate bulk total energy not found. Run total_energy.ipynb for the substrate bulk first."
    )

print(
    f"♻️  Found substrate bulk total energy: {substrate_te_property['_id']} -> "
    f"{substrate_te_property['data']['value']}"
)
display_JSON(substrate_te_property, level=2)


### 4.2. Check film bulk total energy


In [ ]:
io_job_unit = fetch_film_te_subworkflow.get_unit_by_name(name="io-bulk-te-job-film")
job_query_template = io_job_unit.input[0]["endpoint_options"]["params"]["query"]
job_projection_template = io_job_unit.input[0]["endpoint_options"]["params"]["projection"]

BULK_MATERIAL_IDS = BULK_FILM_MATERIAL_IDS
job_query = eval(job_query_template, {"__builtins__": {}}, {"BULK_MATERIAL_IDS": BULK_MATERIAL_IDS})
job_projection = eval(job_projection_template, {"__builtins__": {}})
jobs = client.jobs.list(query=job_query, projection=job_projection)
if not jobs:
    raise RuntimeError(
        "Film bulk total energy not found. Run total_energy.ipynb for the film bulk first."
    )

BULK_TE_JOB_ID = jobs[0]["_id"]
io_te_unit = fetch_film_te_subworkflow.get_unit_by_name(name="io-te-bulk-film")
te_query_template = io_te_unit.input[0]["endpoint_options"]["params"]["query"]
te_projection_template = io_te_unit.input[0]["endpoint_options"]["params"]["projection"]

te_query = eval(te_query_template, {"__builtins__": {}}, {"BULK_TE_JOB_ID": BULK_TE_JOB_ID})
te_projection = eval(te_projection_template, {"__builtins__": {}})
props = client.properties.list(query=te_query, projection=te_projection)
film_te_property = props[0] if props else None
if film_te_property is None:
    raise RuntimeError(
        "Film bulk total energy not found. Run total_energy.ipynb for the film bulk first."
    )

print(
    f"♻️  Found film bulk total energy: {film_te_property['_id']} -> "
    f"{film_te_property['data']['value']}"
)
display_JSON(film_te_property, level=2)


## 5. Configure the Interfacial Energy workflow
### 5.1. Select application


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 5.2. Configure workflow and preview it


In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow


def apply_workflow_kgrids(workflow: Workflow, scf_kgrid=None) -> Workflow:
    if scf_kgrid is not None:
        new_context_scf = PointsGridDataProvider(dimensions=scf_kgrid, isEdited=True).get_context_item_data()
        for subworkflow in workflow.subworkflows:
            unit_names = [unit.name for unit in subworkflow.units]
            if "pw_scf" not in unit_names:
                continue
            unit_to_modify_scf = subworkflow.get_unit_by_name(name="pw_scf")
            unit_to_modify_scf.add_context(new_context_scf)
            subworkflow.set_unit(unit_to_modify_scf)
            break
    return workflow


interfacial_workflow.name = MY_WORKFLOW_NAME
interfacial_workflow = apply_workflow_kgrids(interfacial_workflow, scf_kgrid=SCF_KGRID)

visualize_workflow(interfacial_workflow)


### 5.3. Save workflow to collection


In [ ]:
saved_interfacial_workflow_response = get_or_create_workflow(client, interfacial_workflow, ACCOUNT_ID)
saved_interfacial_workflow = Workflow.create(saved_interfacial_workflow_response)
print(f"Interfacial Energy workflow ID: {saved_interfacial_workflow.id}")


## 6. Create the compute configuration
### 6.1. Select cluster


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 6.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 7. Create the Interfacial Energy job
### 7.1. Create job with interface and workflow configuration


In [ ]:
from mat3ra.notebooks_utils.job import create_job, wait_for_jobs_to_finish_async
from mat3ra.utils.namespace import dict_to_namespace_recursive

print(f"Interface material: {saved_interface.id}")
print(f"Substrate bulk material: {substrate_bulk_response['_id']}")
print(f"Film bulk material: {film_bulk_response['_id']}")
print(f"Interfacial workflow: {saved_interfacial_workflow.id}")
print(f"Project: {project_id}")

interfacial_job_name = f"{MY_WORKFLOW_NAME} {saved_interface.formula} {timestamp}"
interfacial_job_response = create_job(
    api_client=client,
    materials=[saved_interface],
    workflow=saved_interfacial_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=interfacial_job_name,
    compute=compute.to_dict(),
)

interfacial_job = dict_to_namespace_recursive(interfacial_job_response)
interfacial_job_id = interfacial_job._id
print(f"✅ Interfacial Energy job created successfully: {interfacial_job_id}")
display_JSON(interfacial_job_response)


### 7.2. Submit the Interfacial Energy job and monitor the status


In [ ]:
client.jobs.submit(interfacial_job_id)
print(f"✅ Job {interfacial_job_id} submitted successfully!")


In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [interfacial_job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve results
### 8.1. Retrieve and visualize interfacial energy


In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

interfacial_energy_data = client.properties.get_for_job(
    interfacial_job_id, property_name=PropertyName.scalar.interfacial_energy.value
)
interface_te_data = client.properties.get_for_job(
    interfacial_job_id, property_name=PropertyName.scalar.total_energy.value
)

if interfacial_energy_data:
    visualize_properties(interfacial_energy_data, title="Interfacial Energy")
else:
    print("No 'interfacial_energy' property was returned for the job.")

if interface_te_data:
    visualize_properties(interface_te_data, title="Interface Total Energy")
else:
    print("No interface total energy property was returned for the job.")

print(f"Resolved substrate bulk query: {substrate_query}")
print(f"Substrate bulk material: {substrate_bulk_response['_id']}")
print(f"Substrate bulk total energy: {substrate_te_property['data']['value']}")
print(f"Resolved film bulk query: {film_query}")
print(f"Film bulk material: {film_bulk_response['_id']}")
print(f"Film bulk total energy: {film_te_property['data']['value']}")
print(f"Saved interface material: {saved_interface_response['_id']}")
